# Mellum2.1 12B-A2.5B Thinking — mini coding agent on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Mellum2.1-12B-A2.5B-Thinking-coding-agent.ipynb)

[`JetBrains/Mellum2.1-12B-A2.5B-Thinking`](https://huggingface.co/JetBrains/Mellum2.1-12B-A2.5B-Thinking) is JetBrains' 12B mixture-of-experts reasoning model (~2.5B active, 64 experts / 8 per token, 131K context). It writes a chain of thought inside `<think>...</think>` and then the answer. This notebook is a **mini SWE loop**: embed a tiny failing pytest repo, ask the model for a unified diff, apply it, and rerun the tests (at most two iterations).

The native BF16 checkpoint is about 24 GB and does not fit a free T4. The walkthrough uses the official **Q4_K_M GGUF** (~8.1 GB) from [`JetBrains/Mellum2.1-12B-A2.5B-Thinking-GGUF`](https://huggingface.co/JetBrains/Mellum2.1-12B-A2.5B-Thinking-GGUF) through `llama-cpp-python`.

**Licence:** Apache 2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has 16 GB of VRAM, **no native bfloat16**, and **no FlashAttention-2**. Do not load BF16 or turn FlashAttention on. This notebook keeps `flash_attn=False` (llama.cpp's default path, the GGUF equivalent of SDPA / eager) and allocates **32,768** tokens of context so a short repo plus a thinking trace still fit. 16K also works; the trained 131K window is not allocated. Weights plus a 16–32K cache land around 9–11 GB on 15 GB.

**Links:** [model card](https://huggingface.co/JetBrains/Mellum2.1-12B-A2.5B-Thinking) · [GGUF](https://huggingface.co/JetBrains/Mellum2.1-12B-A2.5B-Thinking-GGUF) · [blog](https://blog.jetbrains.com/ai/2026/10/mellum2-1-gets-to-work-a-fast-open-model-for-coding-agents/)


## 1. Install the CUDA `llama-cpp-python` wheel

Remove any CPU build and install the pinned CUDA 12.1 wheel from the project's GitHub releases. A direct wheel URL stops pip from grabbing the CPU package on PyPI. CUDA 12.1 binaries work with the newer driver on a Colab T4. `pytest` is the test runner the agent loop calls.

Mellum's architecture landed in llama.cpp in June 2026; **0.3.36** (1 October 2026) is a wheel after that change.


In [ ]:
%pip uninstall -qqq -y llama-cpp-python
%pip install -qqq --no-cache-dir https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.36-cu121/llama_cpp_python-0.3.36-py3-none-manylinux_2_35_x86_64.whl
%pip install -qqq pytest

# Direct wheel URL, same pattern as colab/LFM-2_5-2_6b-llamacpp-python.ipynb.
# 0.3.36 is after llama.cpp gained the Mellum architecture (PR ggml-org/llama.cpp#23966).


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Native libraries already imported cannot be replaced in the same process. If you skip this after the install, the next cell stops with a version check.


## 2. Check the GPU and print T4 caveats


In [ ]:
import llama_cpp
import torch
from packaging.version import Version

assert Version(llama_cpp.__version__) >= Version("0.3.36"), (
    f"llama-cpp-python {llama_cpp.__version__} is loaded; Mellum GGUF needs 0.3.36+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

!nvidia-smi

major, minor = torch.cuda.get_device_capability()
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
name = torch.cuda.get_device_name()

print()
print("=== T4-first caveats ===")
print(f"GPU: {name}  sm_{major}{minor}  VRAM: {vram_gb:.1f} GB")
print(f"llama-cpp-python {llama_cpp.__version__}")
print("- No native bfloat16 on T4 (sm_75). This notebook loads Q4_K_M GGUF, not BF16.")
print("- Native BF16 weights are ~24 GB and will OOM a 16 GB card.")
print("- No FlashAttention-2 on sm_75. flash_attn=False (sdpa / eager equivalent).")
print("- Official Q4_K_M GGUF is ~8.1 GB; 16-32K context still fits on 15 GB.")
print("- The trained 131K window is not allocated. n_ctx=32768 (16K is also fine).")
print("- MXFP4_MOE (~7.0 GB) is the smaller official quant if you need more KV room.")
print("- Thinking traces can spend thousands of tokens; generation is capped below.")
if major < 8:
    print("- FlashAttention stays off on this GPU.")


## 3. Load the official Q4_K_M GGUF

`Llama.from_pretrained` downloads `Mellum2.1-12B-A2.5B-Thinking-Q4_K_M.gguf` (~8.1 GB) from the JetBrains GGUF repo. `n_gpu_layers=-1` offloads every layer to the T4. `n_ctx=32768` leaves room for the toy repo, a thinking block, and a patch without touching the 131K maximum. `flash_attn=False` is required on sm_75.

Sampler defaults below match the [GGUF card](https://huggingface.co/JetBrains/Mellum2.1-12B-A2.5B-Thinking-GGUF): temperature 0.6, top_p 0.95, top_k 20.


In [ ]:
import os

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

from llama_cpp import Llama

REPO_ID = "JetBrains/Mellum2.1-12B-A2.5B-Thinking-GGUF"
FILENAME = "Mellum2.1-12B-A2.5B-Thinking-Q4_K_M.gguf"
N_CTX = 32768

# Card sampler. Keep these next to the Llama so later cells reuse them.
SAMPLER = dict(temperature=0.6, top_p=0.95, top_k=20)

llm = Llama.from_pretrained(
    repo_id=REPO_ID,
    filename=FILENAME,
    n_gpu_layers=-1,
    n_ctx=N_CTX,
    flash_attn=False,
    verbose=True,
)

print()
print("loaded", FILENAME, "| n_ctx", N_CTX, "| flash_attn False")
print("sampler", SAMPLER)
print()
print("GPU status after load:")
!nvidia-smi


## 4. Smoke test

A short coding question, streamed so the `<think>` block is visible. This is only a sanity check that the GGUF and chat template work before the agent loop.


In [ ]:
def chat(messages, max_tokens=1024, stream=True):
    """One Mellum turn. Returns (think, answer, raw)."""
    kwargs = dict(
        messages=messages,
        max_tokens=max_tokens,
        stream=stream,
        **SAMPLER,
    )
    if not stream:
        raw = llm.create_chat_completion(**kwargs)["choices"][0]["message"]["content"] or ""
        print(raw)
    else:
        parts = []
        for chunk in llm.create_chat_completion(**kwargs):
            delta = chunk["choices"][0]["delta"].get("content") or ""
            if delta:
                parts.append(delta)
                print(delta, end="", flush=True)
        print()
        raw = "".join(parts)

    if "</think>" in raw:
        think, answer = raw.split("</think>", 1)
        think = think.split("<think>", 1)[-1].strip()
        answer = answer.strip()
    else:
        think, answer = "", raw.strip()
    return think, answer, raw


_, answer, _ = chat(
    [{"role": "user", "content": "In one sentence, what does a unified git diff describe?"}],
    max_tokens=512,
)
print()
print("--- answer ---")
print(answer)


## 5. Embed a tiny failing repo

A three-function stats module with intentional bugs (the `mean` bug is the example on the model card). `pytest` fails before the agent runs. Everything lives under `/content/toy-calc` so the notebook does not depend on a network clone.


In [ ]:
import sys
from pathlib import Path
import subprocess
import textwrap

REPO = Path("/content/toy-calc")
(REPO / "tests").mkdir(parents=True, exist_ok=True)

(REPO / "README.md").write_text(
    "# toy-calc\n\nTiny stats helpers. `pytest` in this directory is the contract.\n"
)

(REPO / "stats.py").write_text(
    textwrap.dedent(
        """
        # Tiny stats helpers. Several functions have intentional bugs.


        def mean(xs):
            # Arithmetic mean of a non-empty sequence.
            return sum(xs) / len(xs) - 1


        def clamp(x, lo, hi):
            # Constrain x to the closed interval [lo, hi].
            if x < lo:
                return hi
            if x > hi:
                return hi
            return x


        def running_sum(xs):
            # Prefix sums of xs.
            out = []
            total = 0
            for x in xs:
                out.append(total)
                total += x
            return out
        """
    ).lstrip()
)

(REPO / "tests" / "test_stats.py").write_text(
    textwrap.dedent(
        """
        import sys
        from pathlib import Path

        sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

        from stats import clamp, mean, running_sum


        def test_mean():
            assert mean([2, 4, 6]) == 4.0
            assert mean([10]) == 10.0


        def test_clamp():
            assert clamp(-3, 0, 10) == 0
            assert clamp(15, 0, 10) == 10
            assert clamp(5, 0, 10) == 5


        def test_running_sum():
            assert running_sum([1, 2, 3]) == [1, 3, 6]
            assert running_sum([]) == []
        """
    ).lstrip()
)

print("repo tree:")
for path in sorted(p for p in REPO.rglob("*") if p.is_file()):
    print(" ", path.relative_to(REPO), f"({path.stat().st_size} bytes)")

print()
print("=== pytest before the agent ===")
before = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=REPO,
    text=True,
    capture_output=True,
)
print(before.stdout)
print(before.stderr)
print("exit", before.returncode)


## 6. Patch helpers

The model is asked for a unified diff. These helpers strip the `<think>` block, pull the first `diff` / `patch` fence (or a bare `---` / `+++` hunk), and apply it with `patch`.


In [ ]:
import re
import subprocess
import sys
from pathlib import Path


def extract_unified_diff(text: str) -> str:
    fenced = re.search(r"```(?:diff|patch|udiff)\s*\n(.*?)```", text, flags=re.S)
    if fenced:
        return fenced.group(1).strip() + "\n"
    bare = re.search(r"(?m)^(---\s+\S+[\s\S]+)$", text)
    if not bare:
        raise ValueError("no unified diff found in the model output")
    body = bare.group(1)
    if "```" in body:
        body = body.split("```", 1)[0]
    return body.strip() + "\n"


def extract_stats_file(text: str) -> str | None:
    # Fallback if the model rewrites stats.py instead of emitting a diff.
    for match in re.finditer(r"```(?:python|py)?\s*\n(.*?)```", text, flags=re.S):
        body = match.group(1)
        if "def mean(" in body and "def clamp(" in body and "def running_sum(" in body:
            return body if body.endswith("\n") else body + "\n"
    return None


def apply_patch(diff_text: str, repo: Path) -> subprocess.CompletedProcess:
    # Prefer git-style -p1 (a/stats.py); fall back to -p0 (stats.py). Fuzz helps
    # when the think-trace slightly mis-copies a comment line.
    last = None
    for fuzz in (None, "3"):
        extra = [] if fuzz is None else [f"--fuzz={fuzz}"]
        for strip in ("1", "0"):
            result = subprocess.run(
                ["patch", f"-p{strip}", "--forward", "--batch", *extra],
                cwd=repo,
                input=diff_text,
                text=True,
                capture_output=True,
            )
            last = result
            label = f"patch -p{strip}" + (f" --fuzz={fuzz}" if fuzz else "")
            if result.returncode == 0:
                print("applied with", label)
                print(result.stdout)
                return result
            print(f"{label} failed:\n{result.stdout}{result.stderr}")
    return last


def run_pytest(repo: Path) -> subprocess.CompletedProcess:
    return subprocess.run(
        [sys.executable, "-m", "pytest", "-q"],
        cwd=repo,
        text=True,
        capture_output=True,
    )


## 7. Mini SWE loop (at most two iterations)

The prompt gives the model the repo snapshot and the first pytest failure. It must answer with a unified diff. After `patch` + `pytest`, a failing second turn gets the new file and the new failure. Two iterations is the cap so a long think on T4 stays inside a Colab session.

`max_tokens=4096` leaves room for a think block and a short patch inside the 32K context.


In [ ]:
MAX_ITERS = 2
MAX_TOKENS = 4096


def snapshot(repo: Path) -> str:
    chunks = []
    for path in sorted(p for p in repo.rglob("*") if p.is_file()):
        rel = path.relative_to(repo).as_posix()
        chunks.append(f"### {rel}\n```\n{path.read_text()}\n```")
    return "\n\n".join(chunks)


first_fail = before.stdout + before.stderr
system = (
    "You are a coding agent working in a local git-less checkout. "
    "Read the failing tests, find the bugs, and return a unified diff that makes pytest pass. "
    "Paths in the diff are relative to the repo root (stats.py, not a/content/...). "
    "After thinking, the only thing in your answer should be a unified diff "
    "(optionally inside a ```diff fence). Do not write a prose recap."
)
user = (
    f"Repo root: {REPO}\n\n"
    f"{snapshot(REPO)}\n\n"
    "pytest -q failed with:\n"
    f"```\n{first_fail}\n```\n\n"
    "Propose one unified diff that fixes every failing test."
)

messages = [
    {"role": "system", "content": system},
    {"role": "user", "content": user},
]

passed = False
for step in range(1, MAX_ITERS + 1):
    print(f"\n========== agent iteration {step}/{MAX_ITERS} ==========\n")
    think, answer, raw = chat(messages, max_tokens=MAX_TOKENS)
    messages.append({"role": "assistant", "content": raw})
    print()
    print(f"--- think ({len(think.split())} words) ---")
    preview = think if len(think) <= 800 else think[:800] + "\n..."
    print(preview)
    print()
    print("--- answer ---")
    print(answer)

    try:
        diff = extract_unified_diff(answer)
    except ValueError as exc:
        rewritten = extract_stats_file(answer)
        if rewritten is None:
            print("could not parse a diff:", exc)
            messages.append(
                {
                    "role": "user",
                    "content": (
                        "No unified diff was found. Reply with only a unified diff for stats.py, "
                        "starting with --- / +++ hunks."
                    ),
                }
            )
            continue
        print("no diff; writing a full stats.py rewrite from a python fence")
        (REPO / "stats.py").write_text(rewritten)
        result = run_pytest(REPO)
        print()
        print("=== pytest after rewrite ===")
        print(result.stdout)
        print(result.stderr)
        print("exit", result.returncode)
        if result.returncode == 0:
            passed = True
            break
        messages.append(
            {
                "role": "user",
                "content": (
                    "pytest still fails.\n\n"
                    f"{snapshot(REPO)}\n\n"
                    "pytest -q:\n"
                    f"```\n{result.stdout}{result.stderr}\n```\n"
                    "Reply with a unified diff only."
                ),
            }
        )
        continue

    print()
    print("--- extracted diff ---")
    print(diff)
    applied = apply_patch(diff, REPO)
    if applied.returncode != 0:
        messages.append(
            {
                "role": "user",
                "content": (
                    "patch failed:\n"
                    f"```\n{applied.stdout}{applied.stderr}\n```\n"
                    "Reply with a corrected unified diff only."
                ),
            }
        )
        continue

    result = run_pytest(REPO)
    print()
    print("=== pytest after patch ===")
    print(result.stdout)
    print(result.stderr)
    print("exit", result.returncode)
    if result.returncode == 0:
        passed = True
        break

    messages.append(
        {
            "role": "user",
            "content": (
                "pytest still fails.\n\n"
                f"{snapshot(REPO)}\n\n"
                "pytest -q:\n"
                f"```\n{result.stdout}{result.stderr}\n```\n"
                "Reply with another unified diff only."
            ),
        }
    )

if passed:
    print("\nagent resolved the suite.")
else:
    print(f"\nagent did not clear the suite in {MAX_ITERS} iterations.")

print()
print("=== stats.py after the loop ===")
print((REPO / "stats.py").read_text())
